In [25]:
import pandas as pd
import numpy as np

# --- Archivos base del proyecto ---
athletes_db = "athletes.xlsx"           # Excel: perfil de atletas
activities_db = "activities_running.csv"  # CSV: actividades de running ya filtradas

# Cargamos el Excel con la info de los Atletas

In [26]:
# --- 1. Cargar athletes.xlsx ---
athletes = pd.read_excel(athletes_db)

athlete_cols = ["id", "age", "gender", "activities", "run", "weightkg", " weightstd"]
athletes_clean = athletes[athlete_cols]

print("ATHLETES (columnas relevantes para running)")
print("Shape:", athletes_clean.shape)
print(athletes_clean.columns.tolist())
print(athletes_clean.describe().T)

ATHLETES (columnas relevantes para running)
Shape: (6043, 7)
['id', 'age', 'gender', 'activities', 'run', 'weightkg', ' weightstd']
             count        mean         std    min   25%    50%         75%  \
age         6043.0   34.556677   76.053657 -965.0  28.0   40.0   49.000000   
activities  6043.0  396.844945  552.926931    0.0  54.0  206.0  523.500000   
run         6043.0   39.476750  135.857175    0.0   0.0    0.0   12.000000   
weightkg    5951.0   73.919345   16.941743    1.0  67.0   73.0   79.800342   
 weightstd  5901.0    1.107491   31.845736    0.0   0.0    0.0    0.182237   

                     max  
age          2020.000000  
activities  10476.000000  
run          2612.000000  
weightkg      700.000000  
 weightstd   2399.138768  


Detectamos que hay datos que no pueden ser reales como edades negativas o pesos muy altos

In [27]:
# --- Definir rangos plausibles ---
print("age < 10 o > 100:", ((athletes_clean["age"] < 10) | (athletes_clean["age"] > 100)).sum())
print("weightkg < 30 o > 200:", ((athletes_clean["weightkg"] < 30) | (athletes_clean["weightkg"] > 200)).sum())
print("weightstd < 0 o > 30:", ((athletes_clean[" weightstd"] < 0) | (athletes_clean[" weightstd"] > 30)).sum())

age < 10 o > 100: 545
weightkg < 30 o > 200: 8
weightstd < 0 o > 30: 22


Para el gran volumen (27855) de datos con fechas incorrectas analizamos y vemos que la mayoria son edades que parece que se ha colado un negativo delante.

In [28]:
# --- Investigar el patrón de la columna age ---
print("Valores de age <= 0:")
print(athletes_clean.loc[athletes_clean["age"] <= 0, "age"].value_counts().head(10))

print("\nValores de age > 100:")
print(athletes_clean.loc[athletes_clean["age"] > 100, "age"].value_counts().sort_index().tail(20))

Valores de age <= 0:
age
-70    27
-53    23
-64    22
-58    21
-48    19
-52    18
-62    18
-51    16
-54    16
-47    15
Name: count, dtype: int64

Valores de age > 100:
age
127     1
2020    7
Name: count, dtype: int64


Correjimos los valores negativos por positivos y con el prefijo "-9".

Tambien los que parece que indicaron el año de nacimiento en vez de la edad

In [29]:
# Caso 1: valores tipo -9XX (prefijo "-9" + edad real de 2 dígitos)
mask_9prefix = athletes_clean["age"].between(-999, -900)
athletes_clean.loc[mask_9prefix, "age"] = athletes_clean.loc[mask_9prefix, "age"].abs() % 100
print("Filas corregidas (prefijo -9):", mask_9prefix.sum())

# Caso 2: el resto de negativos (simplemente quitar el signo)
mask_negative_normal = athletes_clean["age"] < 0
athletes_clean.loc[mask_negative_normal, "age"] = athletes_clean.loc[mask_negative_normal, "age"].abs()
print("Filas corregidas (negativo simple):", mask_negative_normal.sum())

# --- 2. Corregir año de nacimiento -> edad real usando el año de la actividad ---
mask_yob = (athletes_clean["age"] >= 1900) & (athletes_clean["age"] <= 2020)
athletes_clean.loc[mask_yob, "age"] = athletes_clean.loc[mask_yob, "age"] - athletes_clean.loc[mask_yob, "age"]
print("Filas corregidas (año de nacimiento):", mask_yob.sum())

print(athletes_clean["age"].describe())

Filas corregidas (prefijo -9): 2
Filas corregidas (negativo simple): 509
Filas corregidas (año de nacimiento): 7
count    6043.000000
mean       41.990733
std        13.467577
min         0.000000
25%        33.000000
50%        42.000000
75%        51.000000
max       127.000000
Name: age, dtype: float64


Lo que siga siendo imposible tras las correcciones, se elimina

In [30]:
mask_age_imposible = (athletes_clean["age"] < 10) | (athletes_clean["age"] > 100)
print("Filas a eliminar tras correcciones:", mask_age_imposible.sum())
athletes_clean = athletes_clean[~mask_age_imposible]
print("Shape final de athletes_clean:", athletes_clean.shape)

Filas a eliminar tras correcciones: 35
Shape final de athletes_clean: (6008, 7)


Para el peso hacemos de la misma manera, y eliminamos la columna de la desviacion, que no nnos interesa para el análisis

In [31]:
# --- weightkg: aplicar mismo rango plausible que en final_df ---
mask_weight_imposible = (athletes_clean["weightkg"] < 30) | (athletes_clean["weightkg"] > 200)
print("Filas a eliminar por weightkg imposible:", mask_weight_imposible.sum())

athletes_clean = athletes_clean[~mask_weight_imposible]

# --- Eliminar columna weightstd por completo ---
athletes_clean = athletes_clean.drop(columns=[" weightstd"], errors="ignore")

print("Shape final de athletes_clean:", athletes_clean.shape)
print("Columnas:", athletes_clean.columns.tolist())
print(athletes_clean.describe())

Filas a eliminar por weightkg imposible: 7
Shape final de athletes_clean: (6001, 6)
Columnas: ['id', 'age', 'gender', 'activities', 'run', 'weightkg']
               age    activities          run     weightkg
count  6001.000000   6001.000000  6001.000000  5912.000000
mean     42.212798    396.024996    39.543576    73.664582
std      13.106037    552.183971   135.893271    10.520146
min      12.000000      0.000000     0.000000    35.600000
25%      33.000000     54.000000     0.000000    67.000000
50%      43.000000    206.000000     0.000000    73.000000
75%      51.000000    522.000000    12.000000    79.797247
max      88.000000  10476.000000  2612.000000   152.912532


# Estimación rápida de volumen de running, tendremos más de 50k filas? Cumplimos de sobras

In [32]:
print("Total de actividades de running en todo el dataset:", athletes["run"].sum())
print("Atletas con al menos 1 actividad de running:", (athletes["run"] > 0).sum())
print(athletes["run"].describe())

Total de actividades de running en todo el dataset: 238558
Atletas con al menos 1 actividad de running: 2589
count    6043.000000
mean       39.476750
std       135.857175
min         0.000000
25%         0.000000
50%         0.000000
75%        12.000000
max      2612.000000
Name: run, dtype: float64


# Cargamos el CSV con las actividades, el número coincide con el contador en la tabla de atletas

In [33]:
# --- 2. Cargar activities_running.csv ---
activities_running = pd.read_csv(activities_db)

print("ACTIVITIES_RUNNING")
print("Shape:", activities_running.shape)
print(activities_running.columns.tolist())
print(activities_running.head())

ACTIVITIES_RUNNING
Shape: (238558, 12)
['id', 'file', 'date', 'age', 'gender', 'sport', 'data', 'workout_time', 'total_distance', 'elevation_gain', 'average_speed', 'average_hr']
                                     id                     file  \
0  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_03_14_44_45.csv   
1  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_05_14_47_02.csv   
2  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_07_11_30_07.csv   
3  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_09_15_41_36.csv   
4  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_12_15_34_22.csv   

                      date  age gender sport             data  workout_time  \
0  2017/01/03 13:44:45 UTC   20      M   Run  TDS-HC-AGL----R        4823.0   
1  2017/01/05 13:47:02 UTC   20      M   Run  TDS-HC-AGL----R        4803.0   
2  2017/01/07 10:30:07 UTC   20      M   Run  TDS-HC-AGL----R        4647.0   
3  2017/01/09 14:41:36 UTC   20      M   Run  TDS-HC-AGL----R        3584.0   
4  2017/01/12 14:

# Hacemos el merge de las dos tablas, la clave es id

In [34]:
final_df = activities_running.merge(
    athletes_clean,
    on="id",
    how="inner",
    suffixes=("_activity", "_athlete")
)

print("Shape final tras merge:", final_df.shape)
print("Columnas:", final_df.columns.tolist())
print(final_df.head())

Shape final tras merge: (237301, 17)
Columnas: ['id', 'file', 'date', 'age_activity', 'gender_activity', 'sport', 'data', 'workout_time', 'total_distance', 'elevation_gain', 'average_speed', 'average_hr', 'age_athlete', 'gender_athlete', 'activities', 'run', 'weightkg']
                                     id                     file  \
0  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_03_14_44_45.csv   
1  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_05_14_47_02.csv   
2  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_07_11_30_07.csv   
3  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_09_15_41_36.csv   
4  273db309-f12f-4520-82ed-0978d5009cc3  2017_01_12_15_34_22.csv   

                      date  age_activity gender_activity sport  \
0  2017/01/03 13:44:45 UTC            20               M   Run   
1  2017/01/05 13:47:02 UTC            20               M   Run   
2  2017/01/07 10:30:07 UTC            20               M   Run   
3  2017/01/09 14:41:36 UTC            20               M

# Shape final tras merge: (237301, 17)

Cumplimos en filas pero no en columnas, vamos a limpiar las columnas duplicadas de los dos db como age y gender y vamos a añadir columnas calculadas importantes para el análisis. (pace, día de la semana, km/semana, etc.)

In [35]:
# --- Limpiar duplicados, me quedo con la data tratada en tabla athletes_clean ---
final_df = final_df.rename(columns={"age_athlete": "age", "gender_athlete": "gender"})
final_df = final_df.drop(columns=["age_activity", "gender_activity"])

print("Shape tras limpiar duplicados:", final_df.shape)
print(final_df.columns.tolist())

Shape tras limpiar duplicados: (237301, 15)
['id', 'file', 'date', 'sport', 'data', 'workout_time', 'total_distance', 'elevation_gain', 'average_speed', 'average_hr', 'age', 'gender', 'activities', 'run', 'weightkg']


# Limpieza de datos

Primero revisamos que las fechas tienen sentido

In [36]:
# --- Convertir date a datetime ---
final_df["date"] = pd.to_datetime(final_df["date"], utc=True)

print("Rango de fechas:", final_df["date"].min(), "a", final_df["date"].max())

# --- Ver cuántas filas tienen fechas sospechosas, se empezó a recoger datos en 2007, así que un date de 1990 es un fallo de reloj del dispositivo, no una actividad real. ---
print("Actividades antes de 2007:", (final_df["date"].dt.year < 2007).sum())
print(final_df["date"].dt.year.value_counts().sort_index())

# --- Quitamos los 147 registros de antes del 2007 ---
final_df = final_df[final_df["date"].dt.year >= 2007]
print("Shape tras quitar fechas imposibles:", final_df.shape)

Rango de fechas: 1990-01-01 08:07:11+00:00 a 2020-05-04 11:31:21+00:00
Actividades antes de 2007: 146
date
1990        1
1999        1
2004       18
2005       46
2006       80
2007      287
2008      290
2009      465
2010     1404
2011     3379
2012     5489
2013     7341
2014    10953
2015    18243
2016    30592
2017    47339
2018    62116
2019    42110
2020     7147
Name: count, dtype: int64
Shape tras quitar fechas imposibles: (237155, 15)


Hacemos una revisión de valores nulos de cada columna

In [37]:
# --- Nulos por columna ---
print(final_df.isnull().sum())
print("\n% de nulos:")
print((final_df.isnull().mean() * 100).round(2))

id                    0
file               4394
date                  0
sport                 0
data                  0
workout_time        777
total_distance     5052
elevation_gain    35314
average_speed      5105
average_hr        44976
age                   0
gender                0
activities            0
run                   0
weightkg              0
dtype: int64

% de nulos:
id                 0.00
file               1.85
date               0.00
sport              0.00
data               0.00
workout_time       0.33
total_distance     2.13
elevation_gain    14.89
average_speed      2.15
average_hr        18.96
age                0.00
gender             0.00
activities         0.00
run                0.00
weightkg           0.00
dtype: float64


Eliminamos los nulos de las columnas "workout_time" y "total_distance", necesarias para calcular el pace.

Eliminamos "average_speed" y la sustituimos por una columna calculada "pace_min_km", para que los datos sean más consistentes.

Para "average_hr" y "elevation_gain" mantenemos NaN (actividades sin ese sensor). No se toca nada, se dejan como están

In [38]:
# --- Eliminar filas sin datos para calcular pace ---
final_df = final_df[final_df["workout_time"].notnull()]
final_df = final_df[final_df["total_distance"].notnull()]

# --- Eliminar average_speed (la sustituimos por pace_min_km, más exacto) ---
final_df = final_df.drop(columns=["average_speed"], errors="ignore")

# --- Comprobación ---
print("Shape final:", final_df.shape)
print(final_df.isnull().sum())

Shape final: (232077, 14)
id                    0
file               3507
date                  0
sport                 0
data                  0
workout_time          0
total_distance        0
elevation_gain    32042
average_hr        42815
age                   0
gender                0
activities            0
run                   0
weightkg              0
dtype: int64


Buscamos actividades duplicadas y las quitamos

In [39]:
# --- Duplicados exactos ---
print("Filas duplicadas exactas:", final_df.duplicated().sum())

# --- Duplicados reales usando id + date ---
print("Duplicados por id + date:", final_df.duplicated(subset=["id", "date"]).sum())

# Los 17 duplicados EXACTOS de fila sí son basura real -> los quitamos
final_df = final_df.drop_duplicates()
print("Shape tras quitar 17 duplicados exactos:", final_df.shape)

Filas duplicadas exactas: 7
Duplicados por id + date: 7
Shape tras quitar 17 duplicados exactos: (232070, 14)


# Empezamos con el análisis de las columnas númericas

In [40]:
# --- Calcular pace en min/km ---
final_df["pace_min_km"] = (final_df["workout_time"] / 60) / final_df["total_distance"]
print(final_df["pace_min_km"].describe())

count    2.320700e+05
mean     2.452151e+02
std      4.717983e+04
min     -2.825737e+03
25%      5.065671e+00
50%      5.655799e+00
75%      6.562194e+00
max      2.123402e+07
Name: pace_min_km, dtype: float64


In [41]:
# --- Analizamos la coherencia de las columnas numericas ---
cols_numericas = ["workout_time", "total_distance", "elevation_gain", "average_hr", "weightkg", "pace_min_km"]
print(final_df[cols_numericas].describe().T)

                   count          mean           std          min  \
workout_time    232070.0  10295.306035  1.123876e+06     1.000000   
total_distance  232070.0      9.565813  1.338249e+02    -0.014480   
elevation_gain  200028.0   1517.738928  1.135992e+05     0.010000   
average_hr      189258.0    144.868748  1.656519e+01     1.000000   
weightkg        232070.0     74.020729  9.429857e+00    38.000000   
pace_min_km     232070.0    245.215094  4.717983e+04 -2825.736648   

                        25%          50%          75%           max  
workout_time    1858.000000  2826.000000  3947.000000  2.541430e+08  
total_distance     5.072400     8.014120    11.385307  4.293580e+04  
elevation_gain    21.623382    54.600000   146.400000  3.324978e+07  
average_hr       136.325112   146.242895   155.288955  2.400000e+02  
weightkg          68.805610    73.200000    79.013590  1.529125e+02  
pace_min_km        5.065671     5.655799     6.562194  2.123402e+07  


Filtramos valores imposibles y posibles outliers

In [42]:
# --- Filas con valores físicamente imposibles ---
print("total_distance <= 0:", (final_df["total_distance"] <= 0).sum())
print("total_distance > 100 km (raro para un run):", (final_df["total_distance"] > 100).sum())
print("workout_time <= 0:", (final_df["workout_time"] <= 0).sum())
print("workout_time > 6h (21600s):", (final_df["workout_time"] > 21600).sum())
print("pace_min_km <= 3 o > 10 min/km:", ((final_df["pace_min_km"] <= 3) | (final_df["pace_min_km"] > 10)).sum())
print("average_hr < 60 o > 220:", ((final_df["average_hr"] < 60) | (final_df["average_hr"] > 220)).sum())

total_distance <= 0: 1
total_distance > 100 km (raro para un run): 149
workout_time <= 0: 0
workout_time > 6h (21600s): 708
pace_min_km <= 3 o > 10 min/km: 15236
average_hr < 60 o > 220: 193


Eliminamos los outliers o valores imposibles (posibles fallos de medición o de sensor)

In [43]:
# --- Detectar registros con valores físicamente imposibles ---
mask_imposibles = (
    (final_df["total_distance"] <= 0) |
    (final_df["total_distance"] > 100) |
    (final_df["workout_time"] <= 0) |
    (final_df["workout_time"] > 21600) |
    (final_df["pace_min_km"] <= 3) | (final_df["pace_min_km"] > 10) |
    (final_df["average_hr"] < 60) | (final_df["average_hr"] > 220)
)

print("Registros a eliminar por valores imposibles:", mask_imposibles.sum())
print("Registros que se conservan:", (~mask_imposibles).sum())

final_df = final_df[~mask_imposibles].copy()
print("Shape tras eliminar valores imposibles:", final_df.shape)

Registros a eliminar por valores imposibles: 15521
Registros que se conservan: 216549


Shape tras eliminar valores imposibles: (216549, 15)


# Nuevas columnas calculadas relevantes para el análisis

A nivel actividad

In [44]:
# Calendario
final_df["year"] = final_df["date"].dt.year
final_df["month"] = final_df["date"].dt.month
final_df["day_of_week"] = final_df["date"].dt.day_name()

# Velocidad derivada del pace (más intuitiva para gráficos)
final_df["pace_kmh"] = 60 / final_df["pace_min_km"]

# Cluster de distancia
final_df["distance_bucket"] = pd.cut(
    final_df["total_distance"],
    bins=[0, 5, 10, 21, float("inf")],
    labels=["corta (<5km)", "media (5-10km)", "larga (10-21km)", "muy larga (>21km)"]
)

print("Columnas tras nivel actividad:", final_df.shape)

Columnas tras nivel actividad: (216549, 20)


Columnas a nivel atletas

In [45]:
athlete_agg = final_df.groupby("id").agg(
    n_activities_run=("pace_min_km", "count"),
    avg_pace=("pace_min_km", "mean"),
    best_pace=("pace_min_km", "min"),
    total_distance_sum=("total_distance", "sum"),
    first_activity=("date", "min"),
    last_activity=("date", "max"),
).reset_index()

# Semanas activas (evitar división por 0)
athlete_agg["weeks_active"] = (
    (athlete_agg["last_activity"] - athlete_agg["first_activity"]).dt.days / 7
).clip(lower=1)

# Km/semana y frecuencia de entrenamiento
athlete_agg["weekly_km"] = athlete_agg["total_distance_sum"] / athlete_agg["weeks_active"]
athlete_agg["training_frequency"] = athlete_agg["n_activities_run"] / athlete_agg["weeks_active"]

# Nivel de experiencia (según nº de actividades de running)
athlete_agg["experience_level"] = pd.cut(
    athlete_agg["n_activities_run"],
    bins=[0, 20, 100, float("inf")],
    labels=["principiante", "intermedio", "avanzado"]
)

# Quitar columnas auxiliares que ya cumplieron su función
athlete_agg = athlete_agg.drop(columns=["first_activity", "last_activity", "weeks_active", "total_distance_sum"])

print("Columnas agregadas por atleta:", athlete_agg.columns.tolist())

Columnas agregadas por atleta: ['id', 'n_activities_run', 'avg_pace', 'best_pace', 'weekly_km', 'training_frequency', 'experience_level']


In [46]:
# Agregar características por atleta a final_df
final_df = final_df.merge(athlete_agg, on="id", how="left")

print("Shape final_df:", final_df.shape)
print("Columnas:", final_df.columns.tolist())
print(final_df.describe().T)

Shape final_df: (216549, 26)
Columnas: ['id', 'file', 'date', 'sport', 'data', 'workout_time', 'total_distance', 'elevation_gain', 'average_hr', 'age', 'gender', 'activities', 'run', 'weightkg', 'pace_min_km', 'year', 'month', 'day_of_week', 'pace_kmh', 'distance_bucket', 'n_activities_run', 'avg_pace', 'best_pace', 'weekly_km', 'training_frequency', 'experience_level']
                       count         mean           std          min  \
workout_time        216549.0  3128.546571   1959.063473     2.000000   
total_distance      216549.0     9.166435      5.712672     0.003620   
elevation_gain      188739.0  1050.678843  42302.680234     0.010000   
average_hr          177702.0   146.388455     14.546608    60.000000   
age                 216549.0    44.168798     11.464709    12.000000   
activities          216549.0  1161.218500    914.038487     1.000000   
run                 216549.0   521.017636    567.978271     1.000000   
weightkg            216549.0    73.907193      9.27

# Al finalizar el EDA, tenemos una db de: 
Shape final_df: (216549, 26)

# Guardamos el dataset final limpio

In [47]:
# --- Exportar dataset final limpio ---
final_df.to_csv("final_dataset.csv", index=False)
print("Guardado como final_dataset.csv")
print("Shape final:", final_df.shape)

Guardado como final_dataset.csv
Shape final: (216549, 26)
